# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez entendido el dataset, vamos a prepararlo para que pueda ser utilizado por algoritmos de Machine Learning.

### Instrucciones Generales:
1. **Solucionar problemas de calidad:** aplicar las decisiones tomadas en el EDA sobre completitud, consistencia, sensibilidad y precisión.
2. **Codificación categórica:** transformar `ocean_proximity` a formato numérico.
3. **Enriquecimiento:** crear variables derivadas útiles para el modelado.
4. **Exportación:** generar un dataset final que será consumido por el Notebook 3.

> **Nota metodológica:** Este notebook deja el dataset **limpio y enriquecido, pero sin escalado**, para evitar fuga de información antes de la separación train/test en el notebook de modelado.


In [ ]:
# Carga de librerías y dataset crudo
import pandas as pd

# Carga del dataset original
datos_crudos = pd.read_csv('../src/data/raw/housing/housing.csv')

print("Dimensiones del dataset crudo:", datos_crudos.shape)
display(datos_crudos.head())


## 1. Solución de problemas de calidad

A partir del EDA se definieron tres acciones principales:
- eliminar 5 registros inconsistentes donde `total_bedrooms > total_rooms`,
- imputar los valores faltantes de `total_bedrooms` con la mediana,
- crear variables flag para registrar topes observados en edad y precio.


In [ ]:
# Eliminar filas inconsistentes donde total_bedrooms > total_rooms
datos_limpios = datos_crudos[
    ~(datos_crudos['total_bedrooms'].notna() & (datos_crudos['total_bedrooms'] > datos_crudos['total_rooms']))
].copy()

print("Dimensiones tras eliminar inconsistencias lógicas:", datos_limpios.shape)


In [ ]:
# Imputar valores faltantes en total_bedrooms utilizando la mediana
mediana_bedrooms = datos_limpios['total_bedrooms'].median()
datos_limpios['total_bedrooms'] = datos_limpios['total_bedrooms'].fillna(mediana_bedrooms)

print("Valores nulos después de imputación:")
display(datos_limpios.isnull().sum())


In [ ]:
# Crear variables flag para registrar topes
valor_max_precio = datos_limpios['median_house_value'].max()

datos_limpios['is_age_capped'] = (datos_limpios['housing_median_age'] >= 52).astype(int)
datos_limpios['is_price_capped'] = (datos_limpios['median_house_value'] >= valor_max_precio).astype(int)

display(datos_limpios[['housing_median_age', 'median_house_value', 'is_age_capped', 'is_price_capped']].head())


### Decisiones documentadas

- **Eliminación de inconsistencias:** se eliminaron 5 filas con una relación ilógica entre habitaciones y dormitorios.
- **Imputación con mediana:** se utilizó la mediana en `total_bedrooms` porque el porcentaje de faltantes era cercano al 1% y la variable puede presentar valores extremos.
- **Flags de topes:** se crearon para conservar la señal de censura observada en el EDA.


## 2. Codificación categórica

La variable `ocean_proximity` es categórica nominal.  
Se utiliza **One-Hot Encoding** porque sus categorías no tienen un orden natural.


In [ ]:
# Aplicar One-Hot Encoding a la columna categórica ocean_proximity
datos_limpios = pd.get_dummies(
    datos_limpios,
    columns=['ocean_proximity'],
    drop_first=True
)

print("Columnas después de One-Hot Encoding:")
display(datos_limpios.columns.tolist())


## 3. Enriquecimiento del dataset

Se crean variables derivadas definitivas para capturar relaciones más informativas que las magnitudes absolutas.


In [ ]:
# Crear variables derivadas definitivas
datos_limpios['rooms_per_household'] = datos_limpios['total_rooms'] / datos_limpios['households']
datos_limpios['rooms_per_person'] = datos_limpios['total_rooms'] / datos_limpios['population']
datos_limpios['bedrooms_per_room'] = datos_limpios['total_bedrooms'] / datos_limpios['total_rooms']

display(datos_limpios[[
    'total_rooms', 'households', 'population', 'total_bedrooms',
    'rooms_per_household', 'rooms_per_person', 'bedrooms_per_room'
]].head())


## 4. Validación final del dataset preparado

Se revisa que el dataset:
- no tenga nulos,
- no conserve la inconsistencia lógica detectada,
- y esté listo para exportación.


In [ ]:
# Validación de nulos
print("Valores nulos por columna:")
display(datos_limpios.isnull().sum().sort_values(ascending=False).head(10))

# Validación de la regla lógica
registros_inconsistentes = datos_limpios[
    datos_limpios['total_bedrooms'] > datos_limpios['total_rooms']
]

print("Registros inconsistentes restantes:", len(registros_inconsistentes))
print("Dimensiones finales del dataset preparado:", datos_limpios.shape)


## 5. Exportación para el Notebook 3

Se guarda el dataset final limpio y enriquecido para ser utilizado en la etapa de modelado.


In [ ]:
# Exportar dataset limpio y enriquecido para el Notebook 3
ruta_salida = '../src/data/processed/housing_preparado.csv'
datos_limpios.to_csv(ruta_salida, index=False)

print(f"Archivo exportado correctamente en: {ruta_salida}")


## 6. Conclusión

El dataset final:
- corrige problemas de completitud y consistencia detectados en el EDA,
- incorpora variables derivadas informativas,
- transforma la variable categórica a formato numérico,
- y queda listo para ser utilizado por el Notebook 3 sin haber aplicado escalado prematuro.
